In [ ]:
import numpy as np
import pandas as pd
import cudaq
import sys
import os
import shutil
import faulthandler
from scipy.linalg import expm
from math import sqrt
from tqdm import tqdm
import torch
sys.path.append(os.path.abspath(".."))
from Utils.qaoaCUDAQ import po_normalize, ret_cov_to_QUBO, qubo_to_ising, process_ansatz_values, kernel_qaoa_X, all_state_to_return, find_budget

In [2]:
e = 0
N_ASSETS = 3
TARGET_QUBIT_IN = 2
q = 1.5
lamb = 0.05

device = torch.device("cuda")
DUPLICATE_ASSET = False
min_P, max_P = 108, 216

In [3]:
data_cov_pd = pd.read_csv("../dataset/top_50_us_stocks_data_20250526_011226_covariance.csv")
data_ret_p_pd = pd.read_csv("../dataset/top_50_us_stocks_returns_price.csv")

data_ret_p_pd = data_ret_p_pd[(data_ret_p_pd["Price"] > min_P) & (data_ret_p_pd["Price"] < max_P)]
data_cov_pd = data_cov_pd.loc[data_cov_pd["Ticker"].isin(data_ret_p_pd["Ticker"])].reset_index(drop=True)
data_cov_pd = data_cov_pd[["Ticker"] + data_cov_pd["Ticker"].tolist()]

In [4]:
np.random.seed(911 + 991 * e + 997 * N_ASSETS)
state = np.random.get_state()
# asset_idx = np.random.choice(data_cov_pd.shape[0], max(TARGET_ASSET), replace=False)
asset_idx = np.random.choice(data_cov_pd.shape[0], N_ASSETS, replace=DUPLICATE_ASSET)
# print(asset_idx)
# asset_idx = np.array([0, 18, 27, 32, 41])
# data_cov = data_cov_pd.drop("Ticker", axis=1)
data_cov = data_cov_pd.drop("Ticker", axis=1).to_numpy()[asset_idx, :][:, asset_idx]
stock_names = data_ret_p_pd["Company_Name"].to_numpy()[asset_idx]
# print("Selected Stocks: ", stock_names)
data_ret_p = data_ret_p_pd.drop("Ticker", axis=1)
# print(data_ret_p.index[asset_idx].to_numpy())
asset_idx_raw = data_ret_p.index[asset_idx].to_numpy()
data_ret_p = data_ret_p.drop("Company_Name", axis=1).to_numpy()[asset_idx, :]

data_ret = data_ret_p[:, 0]
data_p = data_ret_p[:, 1]
print("Selected Stocks: ", stock_names)
print("Selected Stocks Price: ", data_p)
print("Selected Stocks Return: ", data_ret)

Selected Stocks:  ['Philip Morris International Inc.' 'Abbott Laboratories' 'Broadcom Inc.']
Selected Stocks Price:  [158.72999573 132.0383606  167.42999268]
Selected Stocks Return:  [0.00060737 0.00060593 0.00142008]


In [5]:
np.random.set_state(state)
weighted = np.random.uniform(0, 1)
B_mi, B_ma = find_budget(TARGET_QUBIT_IN * N_ASSETS, data_p, min_P, max_P, min_mix_mode=True)
B = B_mi * weighted + B_ma * (1 - weighted)

P = data_p[:N_ASSETS]
ret = data_ret[:N_ASSETS]
cov = data_cov[:N_ASSETS, :N_ASSETS]
P_bb, ret_bb, cov_bb, n_qubit, n_max, C = po_normalize(B, P, ret, cov)
TARGET_QUBIT = n_qubit

# QUBOs of MAX PROBLEM
QU = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, lamb, q)
QU_lamb = ret_cov_to_QUBO(np.zeros_like(ret_bb), np.zeros_like(cov_bb), P_bb, lamb, 0.0)
QU_eval = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, 0.0, q)
QU_return = ret_cov_to_QUBO(ret_bb, np.zeros_like(cov_bb), np.zeros_like(P_bb), 0.0, 0.0)
QU_risk = ret_cov_to_QUBO(np.zeros_like(ret_bb), cov_bb, np.zeros_like(P_bb), 0.0, q)

# Hamiltonians of MIN PROBLEM
H_ansatz = -qubo_to_ising(QU, lamb).canonicalize()
H_lamb = -qubo_to_ising(QU_lamb, lamb).canonicalize()
H_eval = -qubo_to_ising(QU_eval, 0.0).canonicalize()
H_return = -qubo_to_ising(QU_return, 0.0).canonicalize()
H_risk = -qubo_to_ising(QU_risk, 0.0).canonicalize()

In [6]:
n_b = 1<<TARGET_QUBIT
H = H_ansatz.to_matrix()
print("Is Hermitian:", (H == np.conj(H).T).all())
eigvals, eigvecs = np.linalg.eigh(H)

eigvals = np.random.rand(n_b) * 3
eigvals = np.sort(eigvals)
# eigvals = eigvals - eigvals[0]
# eigvals[1:] += 1

H = eigvecs @ np.diag(eigvals) @ eigvecs.T
# print("Modified H:\n", H)

for i in range(len(eigvals)):
    print(f"Eigenvalue {i}: {eigvals[i]}")
    print(f"Eigenvector {i}:\n{eigvecs[:, i]}\n")

ground_state = eigvecs[:, 0]
print("Ground state:\n", ground_state)

# print(np.linalg.eigh(H))
# print(np.linalg.eig(H))

b = np.random.rand(n_b)
b = b / np.linalg.norm(b)
print("Vector b:\n", b)

spectral_gap = eigvals[1] - eigvals[0]
spectral_radius = max(abs(eigvals))
print("\nSpectral gap:", spectral_gap)
print("Spectral radius:", spectral_radius)

ss = spectral_gap / (12 * spectral_radius**3)
F0 = abs(b @ ground_state)**2
print("F0:", F0)
q = 1 - ss * F0 * spectral_gap
print("q:", q)
print("Recommended step size (s):", ss)
assert spectral_radius > 1

Is Hermitian: True
Eigenvalue 0: 0.026456695199172486
Eigenvector 0:
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Eigenvalue 1: 0.0762567911048947
Eigenvector 1:
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j


In [7]:
def commutator(A, B):
    return A @ B - B @ A

In [8]:
# tau = 200
# tau = 0.2
N = 60000
tau = ss * N
# N = 100

print("tau:", tau)


# e^(-tau H)
exp_H = expm(-tau * H)
print("Exponential of -tau * H:\n", exp_H)

result = exp_H @ b
result = result / np.linalg.norm(result)
print("Fidelity with ground state:", abs(result @ ground_state)**2)

print("Result of normed exp(-tau * H) @ b:", result)
# print("Result of normed exp(-tau * H) @ b:", np.round(result))

tau: 9.540502251801792
Exponential of -tau * H:
 [[2.04823043e-10+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 2.46788597e-07+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 0.00000000e+00+0.j 7.62544591e-04+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 ...
 [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  2.67437748e-11+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 1.40157521e-12+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 5.12416601e-13+0.j]]
Fidelity with ground state: 0.5387278975619761
Result of normed exp(-tau * H) @ b: [1.58663795e-10+0.j 2.02008774e-07+0.j 9.65539339e-04+0.j
 1.69875828e-01+0.j 2.27393695e-08+0.j 3.93106557e-05+0.j
 2.31091

In [9]:
s = tau / N

s = 1
N = 20

bb = b.copy()
bb_t = torch.tensor(bb, device=device, dtype=torch.complex64)
H_t = torch.tensor(H, device=device, dtype=torch.complex64)
for i in tqdm(range(N)):
    densi = torch.outer(bb_t, bb_t)
    Q = torch.matrix_exp(s * (densi @ H_t - H_t @ densi))
    bb_t = Q @ bb_t
bb = bb_t.cpu().numpy()
result = bb
print("Fidelity with ground state:", abs(result @ ground_state)**2)
print("Result:", result)
print("Norm:", np.linalg.norm(result))
print(np.round(result))

100%|██████████| 20/20 [00:00<00:00, 76.61it/s]

Fidelity with ground state: 0.42231572730366906
Result: [ 8.3105897e-06+0.j -2.9275004e-22+0.j  7.4516049e-07+0.j
  5.2902713e-02+0.j -3.9859673e-17+0.j  9.0890585e-12+0.j
  6.8746726e-03+0.j  6.3263751e-06+0.j  1.6926478e-12+0.j
  2.7886804e-03+0.j  1.1599208e-04+0.j -1.1081853e-21+0.j
  5.3129159e-03+0.j  2.4237656e-03+0.j  1.1782724e-15+0.j
 -4.2483595e-13+0.j -1.6261772e-24+0.j  6.0760776e-06+0.j
  2.2093593e-01+0.j  3.0770380e-08+0.j  2.2418425e-09+0.j
  1.4297555e-01+0.j  7.6956439e-06+0.j -1.4012985e-45+0.j
  8.4806234e-03+0.j  1.1506781e-04+0.j -6.6441909e-22+0.j
 -6.7579570e-12+0.j  1.5944313e-03+0.j  3.6591373e-16+0.j
 -9.8092519e-13+0.j  7.3956960e-04+0.j  2.3641928e-06+0.j
  2.4543001e-01+0.j  1.3792764e-10+0.j -8.4953897e-18+0.j
  6.4985824e-01+0.j  1.2703282e-06+0.j -1.6300386e-22+0.j
  1.2417865e-06+0.j  4.1846215e-05+0.j -2.6001989e-24+0.j
  2.1909574e-10+0.j  9.1144582e-04+0.j  4.9473038e-21+0.j
 -3.1397563e-12+0.j  2.5439519e-04+0.j  7.2653517e-02+0.j
  9.3506075e-02+

In [50]:
s = tau / N

s = 1
N = 2

bb = b.copy()
# for i in range(N):
#     densi = np.outer(bb, bb)
#     E1 = expm(-1j*sqrt(s)*H)
#     E2 = expm(1j*sqrt(s)*densi)
#     E3 = expm(1j*sqrt(s)*H)
#     bb = np.exp(-1j*sqrt(s)) * E3 @ E2 @ E1 @ bb
#     print(f"iter {i+1}")
#     F_k = abs(bb @ ground_state)**2
#     print(f"F_{i+1}:", F_k)
#     print("bb:", bb)
#     lower_bound = 1 - q**(i)
#     print(f"Lower bound for F_{i+1}:", lower_bound)
#     print()
bb_t = torch.tensor(bb, device=device, dtype=torch.complex64)
H_t = torch.tensor(H, device=device, dtype=torch.complex64)
ground_state_t = torch.tensor(ground_state, device=device, dtype=torch.complex64)
E1 = torch.matrix_exp(-1j*sqrt(s)*H_t)
E3 = torch.matrix_exp(1j*sqrt(s)*H_t)
E1_dagger = torch.adjoint(E1)
for i in tqdm(range(N)):
    densi = torch.outer(bb_t, bb_t)
    E2 = torch.matrix_exp(1j*sqrt(s)*densi)
    E2_dagger = torch.adjoint(E2)
    E321 = E3 @ E2 @ E1
    E321_dagger = torch.adjoint(E321)
    print(torch.trace(E2_dagger @ E2))
    eigvalss, eigvecss = torch.linalg.eig(E321)
    print("Eigenvalues of E321:", torch.sort(torch.abs(eigvalss)))
    bbb = bb_t.clone()
    bb_t = E321 @ bb_t
    if (i+1) % 1 == 0:
        F_k = abs(bb_t @ ground_state_t)**2
        print(f"iter {i+1}")
        print("norm of bb_t:", torch.norm(bb_t).item())
        print(f"F_{i+1}:", F_k)
        lower_bound = 1 - q**(i+1)
        print(f"Lower bound for F_{i+1}:", lower_bound)
        print()
bb = bb_t.cpu().numpy()

print("Fidelity with ground state:", abs(bb @ ground_state)**2)
print("Result of DB-QITE:", bb)
print("Abs of DB-QITE result:", np.abs(bb) * np.sign(bb.real))
print("Norm of DB-QITE result:", np.linalg.norm(bb))

100%|██████████| 2/2 [00:00<00:00, 106.87it/s]

tensor(64.-6.9305e-09j, device='cuda:0')
Eigenvalues of E321: torch.return_types.sort(
values=tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], device='cuda:0'),
indices=tensor([ 3,  6,  9, 13, 27, 30, 39, 46, 53, 48, 19, 36, 59, 16, 26, 12, 34,  0,
        50, 33, 42, 58, 62, 20, 52, 14, 32, 47, 55, 56, 61, 10, 28,  1, 60, 63,
        43, 57, 37, 41, 17, 25,  5, 11, 21, 31, 35, 54, 15, 24, 29, 38, 22, 40,
        44, 49, 18, 45, 23,  7,  8, 51,  2,  4], device='cuda:0

In [11]:
print(torch.trace(E1 @ E3))
print(np.exp(-1j*sqrt(s)))

tensor(64.0000-1.0138e-08j, device='cuda:0')
(0.5403023058681398-0.8414709848078965j)


In [32]:
print(s)

1


In [53]:
# print(densi)
densi = torch.outer(bbb, bbb)
dd = torch.matrix_exp(1j*10*densi)
dd_dagger = torch.adjoint(dd)
print(torch.trace(dd_dagger @ dd))
print(torch.diag(dd_dagger @ dd))

tensor(63.1672+1.1238e-10j, device='cuda:0')
tensor([1.0008-3.9191e-11j, 0.9957+6.9049e-11j, 0.9605-4.6607e-11j,
        0.9221-2.6152e-10j, 0.9996-4.7859e-12j, 0.9875-5.8532e-11j,
        0.9984-3.6426e-12j, 0.9986+6.7735e-12j, 0.9981-3.1549e-12j,
        0.9917+1.0208e-11j, 0.9993+3.8653e-13j, 0.9826+2.8780e-11j,
        0.9571+2.7737e-11j, 0.9564+6.7852e-11j, 0.9867+3.1630e-12j,
        0.9937-5.1023e-11j, 0.9955-6.8433e-11j, 0.9836-5.8549e-11j,
        0.9252+1.1947e-10j, 0.9809-1.5357e-10j, 0.9981-2.5713e-11j,
        0.9340+3.0776e-10j, 0.9822+1.2006e-10j, 0.9961-1.1855e-11j,
        0.9474-2.5204e-11j, 0.9983-9.5540e-13j, 0.9797-5.1208e-11j,
        0.9979+1.0117e-12j, 0.9289-1.7890e-11j, 0.9908-2.4381e-11j,
        0.9957-4.8678e-12j, 1.0117+9.9003e-11j, 0.9986-5.9383e-12j,
        0.9798-1.0941e-10j, 0.9841-1.3231e-10j, 0.9950-5.9826e-12j,
        0.9650+5.1029e-10j, 0.9905-1.3815e-11j, 0.9972+3.5991e-11j,
        1.0001+4.3293e-12j, 0.9555-1.6385e-10j, 0.9898+1.0318e-10j,
   

In [51]:
eig_valss, eig_vecss = torch.linalg.eig(dd)
print(abs(eig_valss))
print(eig_vecss[:, 1])

tensor([1.0000e+00, 4.3706e-04, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00], device='cuda:0')
tensor([0.0319-0.0206j, 0.0692+0.0190j, 0.2012+0.0352j, 0.2830+0.0000j,
        0.0207+0.0049j, 0.1168+0.0295j, 0.0411+0.0001j

In [56]:
print(bbb)

tensor([0.0376-0.0056j, 0.0552+0.0459j, 0.1688+0.1150j, 0.2579+0.1167j,
        0.0168+0.0130j, 0.0942+0.0750j, 0.0374+0.0170j, 0.0317+0.0195j,
        0.0365+0.0292j, 0.0819+0.0410j, 0.0233+0.0127j, 0.1122+0.0967j,
        0.1858+0.0937j, 0.1856+0.0973j, 0.0978+0.0818j, 0.0674+0.0445j,
        0.0566+0.0475j, 0.1096+0.0701j, 0.2598+0.1070j, 0.1168+0.0842j,
        0.0370+0.0273j, 0.2423+0.1023j, 0.1142+0.0726j, 0.0528+0.0451j,
        0.2066+0.1024j, 0.0359+0.0200j, 0.1212+0.1046j, 0.0408+0.0207j,
        0.2354+0.1271j, 0.0813+0.0682j, 0.0563+0.0357j, 0.0753-0.0337j,
        0.0317+0.0201j, 0.1375+0.0535j, 0.1066+0.0827j, 0.0600+0.0475j,
        0.1843+0.0679j, 0.0831+0.0550j, 0.0447+0.0371j, 0.0187-0.0012j,
        0.1816+0.1112j, 0.0857+0.0741j, 0.0130+0.0050j, 0.0148-0.0087j,
        0.0737+0.0633j, 0.0490+0.0291j, 0.0234-0.0107j, 0.0280-0.0223j,
        0.2135+0.0920j, 0.0159+0.0126j, 0.0367+0.0283j, 0.0379-0.0163j,
        0.1727+0.1165j, 0.0617+0.0500j, 0.0608-0.0185j, 0.0447-0